# Train Blueprint Analysis Model with YOLOv8

This notebook is designed to run on Google Colab to train your AI blueprint analysis model using a free cloud GPU.

**Before you start:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Upload your `blueprint_dataset.zip` file to your Google Drive (e.g., inside a folder named `Blueprints`).

In [ ]:
# Install the required Ultralytics package
!pip install ultralytics

## 1. Mount Google Drive
We need to connect your Google Drive so the notebook can read your dataset and save the trained model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Unzip Dataset
We will extract the pre-formatted YOLO dataset directly to the Colab instance for fast reading during training.
**IMPORTANT:** Update `zip_path` if your file is in a different location in your Drive.

In [ ]:
import os
import zipfile

# Change this path if your data is stored elsewhere in your Drive
zip_path = "/content/drive/MyDrive/Blueprints/blueprint_dataset.zip"
extract_dir = "/content/datasets"

print(f"Extracting {zip_path}...")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("Extraction complete!")

## 3. Fix YAML Paths
The `data.yaml` file was generated on Windows and contains a Windows path. We need to update it to the Colab path.

In [ ]:
import yaml

yaml_path = '/content/datasets/blueprints_large/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/datasets/blueprints_large'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("YAML path updated to:", data['path'])

## 4. Train YOLOv8 Model
Now we train the model. We use `yolov8m.pt` (Medium) for high accuracy, and run for 100 epochs.

In [ ]:
from ultralytics import YOLO

# Load a pre-trained Medium model
model = YOLO('yolov8m.pt') 

# Train for 100 epochs using the GPU (device=0)
results = model.train(
    data="/content/datasets/blueprints_large/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='blueprint_model'
)

## 5. Save the Final Model
Once training is finished, we copy the best weights back to your Google Drive so you can download it to your PC.

In [ ]:
import shutil

source_model = "/content/runs/blueprint_model/weights/best.pt"
drive_base_path = "/content/drive/MyDrive/Blueprints"
dest_model = os.path.join(drive_base_path, "best_blueprint_model.pt")

shutil.copy2(source_model, dest_model)
print(f"Model saved to Google Drive at: {dest_model}")
print("You can now download it from Google Drive and replace your local best.pt file!")